Importantions of libraries

In [549]:
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import seaborn as sns
import pandas as pd
import numpy as np
import yfinance as yf
from typing import Optional
import requests
from datetime import datetime, timedelta
from abc import ABC, abstractmethod
import logging
from typing import List, Dict, Optional, Union
from dataclasses import dataclass, field


Class Stocks

In [550]:
@dataclass
class Asset(ABC):
    symbol: str
    _prices: Optional[pd.DataFrame] = None

    @abstractmethod
    def get_prices(self) -> pd.DataFrame:
        pass

    @abstractmethod
    def test_symbol(self) -> str:
        pass

@dataclass
class Crypto(Asset):

    provider: Optional[Provider] = None
    def __post_init__(self):
        if self.provider is None:
            self.provider = YahooProvider()

    def get_prices(self) -> pd.DataFrame:
        if self._prices is None:
            logging.info(f"Fetching prices for {self.symbol}...")
            self._prices = self.provider.get_prices(self.symbol)
        return self._prices

    def test_symbol(self) -> str:
        return self.provider.test_symbol(self.symbol)

    def get_close_series(self) -> pd.Series:
        prices = self.get_prices()
        if 'Close' not in prices.columns:
            raise ValueError("Missing 'Close' column in price data")
        return prices['Close'][self.symbol]

    def get_current_price(self) -> float:
        return self.get_prices()['Close'].iloc[-1]

@dataclass
class Provider(ABC):

    @abstractmethod
    def get_prices(self, symbol: str) -> pd.DataFrame:
        pass

    @abstractmethod
    def test_symbol(self, symbol: str) -> str:
        pass

@dataclass
class YahooProvider(Provider):

    period: str = "1d"
    interval: str = "1m"

    def get_prices(self, symbol: str) -> pd.DataFrame:
        data = yf.download(symbol, period=self.period, interval=self.interval)
        if data.empty:
            raise ValueError(f"Failed to fetch data for {symbol}")
        return data

    def test_symbol(self, symbol: str) -> str:
        try:
            data = self.get_prices(symbol)
            if data.empty:
                raise ValueError
            return symbol
        except Exception:
            raise ValueError(f"Symbol {symbol} not found")

@dataclass
class Indicator(ABC):

    @abstractmethod
    def calculate(self, asset: Asset):
        pass

    def apply_indicator(self, result, indicator=None):
        if indicator is None:
            return result
        return indicator.calculate(result)

    def calculate_with(self, asset, indicator=None):
        result = self.calculate(asset)
        if indicator is not None:
            result = self.apply_indicator(result, indicator)
        return result

@dataclass
class Signal(ABC):

    @abstractmethod
    def generate_signals(self, asset: Asset):
        pass

@dataclass
class SimpleMovingAverage(Indicator):
    period: int = 20

    def calculate(self, asset: Asset | pd.Series) -> pd.Series:
        if hasattr(asset, 'get_close_series'):
            asset = asset.get_close_series()
        result = asset.rolling(window=self.period).mean()
        result.name = f"SMA_{self.period}"
        return result

@dataclass
class ExponentialMovingAverage(Indicator):
    period: int = 20

    def calculate(self, asset: Asset | pd.Series) -> pd.Series:
        if hasattr(asset, 'get_close_series'):
            asset = asset.get_close_series()
        result = asset.ewm(span=self.period, adjust=False).mean()
        result.name = f"EMA_{self.period}"
        return result

@dataclass
class Rsi(Indicator):
    period: int = 14

    def calculate(self, asset: Asset | pd.Series) -> pd.Series:
        if hasattr(asset, 'get_close_series'):
            asset = asset.get_close_series()
        delta = asset.diff()
        gain = (delta.where(delta > 0, 0)).rolling(self.period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(self.period).mean()
        loss = loss.replace(0, np.nan)
        rs = gain / loss
        rsi = 100 - (100 / (1 + rs))
        rsi.name = f"RSI_{self.period}"
        return rsi

@dataclass
class RsiSignal(Signal):
    rsi: Rsi
    buy_threshold: float = 25
    sell_threshold: float = 80

    def generate_signals(self, asset: Asset) -> pd.Series:

        rsi = self.rsi.calculate(asset)
        signals = pd.Series(0, index=rsi.index)
        signals[(rsi < self.buy_threshold)] = -1 #Buy
        signals[(rsi > self.sell_threshold)] = 1 #Sell

        return signals

@dataclass
class StochRsi(Indicator):
    period: int = 14
    smooth: int = 3
    rsi: Rsi = field(default_factory=Rsi)

    def calculate(self, asset: Asset | pd.Series) -> pd.DataFrame:

        rsi_series = self.rsi.calculate(asset)
        rsi_low = rsi_series.rolling(self.period).min()
        rsi_high = rsi_series.rolling(self.period).max()
        denom = (rsi_high - rsi_low).replace(0, np.nan)
        k_raw = (rsi_series - rsi_low) / denom * 100
        k = k_raw.rolling(window=self.smooth).mean() if self.smooth and self.smooth > 1 else k_raw
        d = k.rolling(window=self.smooth).mean() if self.smooth and self.smooth > 1 else k
        k.name = f"Stoch_RSI_K_{self.period}_{self.smooth}"
        d.name = f"Stoch_RSI_D_{self.period}_{self.smooth}"
        return pd.DataFrame({
            "rsi": rsi_series,
            "k": k,
            "d": d
            }).dropna()

@dataclass
class StochRsiSignal(Signal):
    stoch: StochRsi
    buy_threshold: float = 5
    sell_threshold: float = 95

    def generate_signals(self, asset: Asset) -> pd.Series:

        stoch = self.stoch.calculate(asset)
        signals = pd.Series(0, index=stoch.index)
        signals[(stoch['d'] < self.buy_threshold)] = -1 #Buy
        signals[(stoch['d'] > self.sell_threshold)] = 1 #Sell

        return signals

@dataclass
class BollingerBands(Indicator):
    window: int = 20
    std_dev: int = 2

    def calculate(self, asset: Asset | pd.Series) -> pd.Series:

        if hasattr(asset, 'get_close_series'):
            asset = asset.get_close_series()
        price = asset
        middle = price.rolling(self.window).mean()
        std = price.rolling(self.window).std()
        upper = middle + self.std_dev * std
        lower = middle - self.std_dev * std
        bb_percent_b = (price - lower) / (upper - lower)
        bb_percent_b.name = f"BB_percent_b_{self.window}_{self.std_dev}"

        return bb_percent_b

@dataclass
class BollingerBandsSignal(Signal):
    bb: BollingerBands
    buy_threshold: float = 0
    sell_threshold: float = 1

    def generate_signals(self, asset: Asset) -> pd.Series:

        bb = self.bb.calculate(asset)
        signals = pd.Series(0, index=bb.index)
        signals[(bb < self.buy_threshold)] = -1 #Buy
        signals[(bb > self.sell_threshold)] = 1 #Sell

        return signals

@dataclass
class Macd(Indicator):
    fast: int = 12
    slow: int = 26
    signal: int = 9

    def calculate(self, asset: Asset | pd.Series) -> pd.DataFrame:
        if hasattr(asset, 'get_close_series'):
            asset = asset.get_close_series()
        close = asset
        exp1 = close.ewm(span=self.fast).mean()
        exp2 = close.ewm(span=self.slow).mean()
        macd = exp1 - exp2
        signal = macd.ewm(span=self.signal).mean()
        histogram = macd - signal
        macd.name = f"MACD_{self.fast}_{self.slow}_{self.signal}"
        signal.name = f"MACD_Signal_{self.fast}_{self.slow}_{self.signal}"
        histogram.name = f"MACD_Histogram_{self.fast}_{self.slow}_{self.signal}"
        return pd.DataFrame({
        "macd": macd,
        "signal": signal,
        "histogram": histogram
        })


@dataclass
class MacdSignal(Signal):
    macd: Macd
    difference = 10

    def generate_signals(self, asset: Asset) -> pd.Series:
        macd = self.macd.calculate(asset)

        macd_line = macd['macd']
        signal_line = macd['signal']

        signals = pd.Series(0, index=macd.index)

        cross_up = (macd_line > signal_line) & (macd_line.shift(1) <= signal_line.shift(1))
        cross_down = (macd_line < signal_line) & (macd_line.shift(1) >= signal_line.shift(1))

        signals[cross_up] = 1
        signals[cross_down] = -1

        return signals

@dataclass
class SharpeRatio(Indicator):
    period: int = 52 # Default period for 1wk
    window: int = 60 # Default window for 1wk
    rf: float = 0.01

    def calculate(self, asset: Asset | pd.Series) -> pd.Series:
        if hasattr(asset, 'get_close_series'):
            asset = asset.get_close_series()
        close_prices = asset
        returns = Calculations.log_return(close_prices)
        sharpe_ratio = Calculations.annualized_sharpe(returns, period=self.period, window=self.window, rf=self.rf)
        sharpe_ratio.name = f"Sharpe_{self.period}_{self.window}_{self.rf}"
        return sharpe_ratio

@dataclass
class SharpeSignal(Signal):
    sharpe: SharpeRatio
    buy_threshold: float = -1.5 #Default buy threshold (Sharpe 1wk)
    sell_threshold: float = 2.0 #Default sell threshold (Sharpe 1wk)

    def generate_signals(self, asset: Asset | pd.Series) -> pd.Series:

        sharpe = self.sharpe.calculate(asset)
        signals = pd.Series(0, index=sharpe.index)
        signals[sharpe < self.buy_threshold] = -1
        signals[sharpe > self.sell_threshold] = 1

        return signals
@dataclass
class SortinoRatio(Indicator):
    period: int = 52 # Default period for 1wk
    window: int = 60 # Default window for 1wk
    rf: float = 0.01

    def calculate(self, asset: Asset | pd.Series) -> pd.Series:
        if hasattr(asset, 'get_close_series'):
            asset = asset.get_close_series()
        close_prices = asset
        returns = Calculations.log_return(close_prices)#modificar aqui se quiser sem log! returns = self.create_dataframe_close().pct_change().dropna()
        sortino_ratio = Calculations.annualized_sortino(returns, period=self.period, window=self.window, rf=self.rf)
        sortino_ratio.name = f"Sortino_{self.period}_{self.window}_{self.rf}"
        return sortino_ratio

@dataclass
class SortinoSignal(Signal):
    sortino: SortinoRatio
    buy_threshold: float = -1.5 # Default buy threshold (Sortino 1wk)
    sell_threshold: float = 4.5 # Default sell threshold (Sortino 1wk)

    def generate_signals(self, asset: Asset | pd.Series) -> pd.Series:

        sortino = self.sortino.calculate(asset)
        signals = pd.Series(0, index=sortino.index)
        signals[sortino < self.buy_threshold] = -1
        signals[sortino > self.sell_threshold] = 1

        return signals

class Calculations:

    @staticmethod
    def log_return(prices: pd.Series | Asset) -> pd.Series:
        if hasattr(prices, 'get_close_series'):
            prices = prices.get_close_series()
        returns = np.log(prices / prices.shift(1)).dropna()
        returns.name = 'log_return'
        return returns

    @staticmethod
    def annualized_sharpe(returns: pd.Series, period=52, rf=0.01, window=60) -> pd.Series:
        """
        Calculate annualized Sharpe ratio with rolling window.

        Args:
            returns: Returns series (daily or other frequency)
            period: Number of periods per year (default 252 for trading days)
            rf: Risk-free rate (default 0.01 = 1%)
            window: Rolling window size for calculation

        Returns:
+            pd.Series: Annualized Sharpe ratio with rolling window
+        """
        rolling_mean = returns.rolling(window=window).mean() * period
        rolling_std = returns.rolling(window=window).std() * np.sqrt(period)
        sharpe_ratio = (rolling_mean - rf) / rolling_std
        sharpe_ratio = sharpe_ratio.replace([np.inf, -np.inf], np.nan)
        return sharpe_ratio

    @staticmethod
    def annualized_sortino(returns: pd.Series, period=52, rf=0.01, window=60) -> pd.Series:
        """
+        Calculate annualized Sortino ratio with rolling window.
+
+        Args:
+            returns: Returns series (daily or other frequency)
+            period: Number of periods per year (default 252 for trading days)
+            rf: Risk-free rate (default 0.01 = 1%)
+            window: Rolling window size for calculation
+
+        Returns:
+            pd.Series: Annualized Sortino ratio with rolling window
+        """
        mar = rf / period
        excess = returns - mar
        downside = np.minimum(0, excess)
        downside_deviation = downside.rolling(window=window).std() * np.sqrt(period)
        rolling_mean = excess.rolling(window=window).mean() * period
        sortino_ratio = rolling_mean / downside_deviation
        sortino_ratio = sortino_ratio.replace([np.inf, -np.inf], np.nan)
        return sortino_ratio

@dataclass
class CombinedSignal:
    signals: list
    weights: list
    quantity_threshold: float = None

    def generate_signals(self, asset: Asset):

        signals_list = [
            signal.generate_signals(asset)
            for signal in self.signals
        ]

        if self.quantity_threshold is None:
            self.quantity_threshold = len(self.signals)

        df = pd.concat(signals_list, axis=1).fillna(0)

        weights = np.array(self.weights)

        combined = (df * weights).sum(axis=1)

        final_signal = pd.Series(0, index=combined.index)
        final_signal[combined >= self.quantity_threshold] = 1
        final_signal[combined <= -self.quantity_threshold] = -1

        return final_signal

@dataclass
class Btc(Crypto):
    pass

In [551]:
#Get BTC data
btc_1_provider = YahooProvider(period="7y", interval="1wk")
btc = Crypto("BTC-USD", provider=btc_1_provider)

In [552]:
#Get VIX data
vix_provider = YahooProvider(period='7y', interval='1wk')
vix = Crypto('^VIX', provider=vix_provider).get_close_series()

#Plot VIX with SMA_9 and SMA_20
vix_sma_9 = ExponentialMovingAverage(period=7).calculate(vix)
vix_sma_20 = ExponentialMovingAverage(period=20).calculate(vix)

[*********************100%***********************]  1 of 1 completed


In [553]:
sma_200 = SimpleMovingAverage(period=200)
sma_100 = SimpleMovingAverage(period=100)
sma_50 = SimpleMovingAverage(period=50)
sma_20 = SimpleMovingAverage(period=20)

ema_200 = ExponentialMovingAverage(period=200)
ema_100 = ExponentialMovingAverage(period=100)
ema_50 = ExponentialMovingAverage(period=50)

btc_sma_200 = sma_200.calculate(btc)
btc_sma_100 = sma_100.calculate(btc)
btc_sma_50 = sma_50.calculate(btc)

[*********************100%***********************]  1 of 1 completed


In [554]:
btc_ema_200 = ema_200.calculate(btc)
btc_ema_100 = ema_100.calculate(btc)
btc_ema_50 = ema_50.calculate(btc)


In [555]:
btc_rsi = Rsi(period=14) #Used for plot
btc_rsi_values = btc_rsi.calculate(btc)
btc_rsi_signals = RsiSignal(rsi=btc_rsi).generate_signals(btc) #Used for signals
rsi_sma_20 = SimpleMovingAverage(period=20).calculate(btc_rsi_values) #Used for plot SMA_20
btc_rsi_buy = btc_rsi_signals == -1
btc_rsi_sell = btc_rsi_signals == 1

In [556]:
btc_stochrsi = StochRsi(period=14, smooth=3) #Used for plot
btc_stochrsi_values = btc_stochrsi.calculate(btc)
btc_stochrsi_signals = StochRsiSignal(stoch=btc_stochrsi).generate_signals(btc)# Used for signals
btc_stochrsi_buy = btc_stochrsi_signals == -1
btc_stochrsi_sell = btc_stochrsi_signals == 1

In [557]:
btc_bb = BollingerBands(window=30, std_dev=2) #Used for plot
btc_bb_values = btc_bb.calculate(btc)
btc_bb_signals = BollingerBandsSignal(bb=btc_bb).generate_signals(btc) #Used for signals
bb_sma_20 = SimpleMovingAverage(period=20).calculate(btc_bb_values) #Used for plot SMA_20
btc_bb_signals.value_counts() #Used for count signals
btc_bb_buy = btc_bb_signals == -1
btc_bb_sell = btc_bb_signals == 1

In [558]:
btc_macd = Macd()#Used for plot
btc_macd_values = btc_macd.calculate(btc)
btc_macd_signals = MacdSignal(btc_macd).generate_signals(btc)#Used for signals
btc_macd_buy = btc_macd_signals == 1
btc_macd_sell = btc_macd_signals == -1

In [559]:
#Used class CombinedSignal
btc_sharpe = SharpeRatio(period=52, window=60)#Used for plot
btc_sortino = SortinoRatio(period=52, window=60)
btc_df = btc.get_prices()

btc_combined = CombinedSignal(signals = [SharpeSignal(sharpe=btc_sharpe), SortinoSignal(sortino=btc_sortino, buy_threshold=-1.7, sell_threshold=5.65)], weights = [1.0, 1.0], quantity_threshold=2.0)
btc_combined_signals = btc_combined.generate_signals(btc)
btc_combined_signals = btc_combined_signals.reindex(btc_df.index).fillna(0) #Used fo plot in BTC-USD candlestick

buy_sharpe_sortino = btc_combined_signals == -1
sell_sharpe_sortino = btc_combined_signals == 1

In [560]:
#Used class SharpeSignal Default Settings
btc_sharpe = SharpeRatio(period=52, window=60)
btc_sharpe_values = btc_sharpe.calculate(btc)
btc_sharpe_signals = SharpeSignal(sharpe=btc_sharpe).generate_signals(btc)

btc_sharpe_sma_20 = SimpleMovingAverage(period=20).calculate(btc_sharpe_values)#Used for plot SMA_20 in sharpe

sharpe_buy = btc_sharpe_signals == -1
sharpe_sell = btc_sharpe_signals == 1

In [561]:
#Used class SortinoSignal Default Settings
btc_sortino = SortinoRatio(period=52, window=60)
btc_sortino_values = btc_sortino.calculate(btc)
btc_sortino_signals = SortinoSignal(sortino=btc_sortino).generate_signals(btc)

btc_sortino_sma_14 = SimpleMovingAverage(period=14).calculate(btc_sortino_values)#Used for plot SMA_20 in sortino
btc_sortino_sma_7 = SimpleMovingAverage(period=7).calculate(btc_sortino_values)#Used for plot SMA_9 in sortino

sortino_buy = btc_sortino_signals == -1
sortino_sell = btc_sortino_signals == 1

In [562]:
fig2 = make_subplots(
    rows=1, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1,
    subplot_titles=('BTC-USD'),
    row_width=[0.2]
)

fig2.add_trace(go.Candlestick(
    x=btc_df.index,
    open=btc_df['Open']['BTC-USD'],
    high=btc_df['High']['BTC-USD'],
    low=btc_df['Low']['BTC-USD'],
    close=btc_df['Close']['BTC-USD'],
    name='BTC-USD',

), row=1, col=1)

fig2.add_trace(go.Scatter(
    x=btc_df.index[buy_sharpe_sortino],
    y=btc_df['Close']['BTC-USD'][buy_sharpe_sortino],
    mode='markers',
    marker=dict(size=15,
                color='green',
                colorscale='PuRd',
                symbol='triangle-up'),
    opacity=0.7,
    showlegend=True,
    name='Buy Sharpe-Sortino Signals'
), row=1, col=1)

fig2.add_trace(go.Scatter(
    x=btc_df.index[sell_sharpe_sortino],
    y=btc_df['Close']['BTC-USD'][sell_sharpe_sortino],
    mode='markers',
    marker=dict(size=15,
                color='red',
                colorscale='PuRd',
                symbol='triangle-down'),
    opacity=0.7,
    showlegend=True,
    name='Sell Sharpe-Sortino Signals'
), row=1, col=1)

fig2.update_layout(xaxis_rangeslider_visible=False, template='plotly_dark')

In [563]:
fig = make_subplots(
    rows=7, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.0008,
    subplot_titles=('BTC-USD'),
    row_width=[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 2.0]
)

fig.add_trace(go.Candlestick(
    x=btc_df.index,
    open=btc_df['Open']['BTC-USD'],
    high=btc_df['High']['BTC-USD'],
    low=btc_df['Low']['BTC-USD'],
    close=btc_df['Close']['BTC-USD'],
    name='BTC-USD',

), row=1, col=1)

fig.add_trace(go.Scatter(
    x=btc_df.index[buy_sharpe_sortino],
    y=btc_df['Close']['BTC-USD'][buy_sharpe_sortino],
    mode='markers',
    marker=dict(size=15,
                color='green',
                colorscale='PuRd',
                symbol='triangle-up'),
    opacity=0.7,
    showlegend=True,
    name='Buy Sharpe-Sortino Signals'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=btc_df.index[sell_sharpe_sortino],
    y=btc_df['Close']['BTC-USD'][sell_sharpe_sortino],
    mode='markers',
    marker=dict(size=15,
                color='red',
                colorscale='PuRd',
                symbol='triangle-down'),
    opacity=0.7,
    showlegend=True,
    name='Sell Sharpe-Sortino Signals'
), row=1, col=1)

fig.update_layout(xaxis_rangeslider_visible=False, template='plotly_dark', height=1150, width=1600)

In [564]:
fig.add_trace(go.Scatter(
    x=btc_rsi_values.index,
    y=btc_rsi_values,
    line=dict(color='lightblue', width=2),
    showlegend=True,
    name='RSI'
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=btc_rsi_values[btc_rsi_buy].index,
    y=btc_rsi_values[btc_rsi_buy],
    mode='markers',
    marker=dict(size=10,
                color='green',
                symbol='triangle-up'),
    showlegend=True,
    opacity=0.7,
    name='RSI Buy Signals'
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=btc_rsi_values[btc_rsi_sell].index,
    y=btc_rsi_values[btc_rsi_sell],
    mode='markers',
    marker=dict(size=10,
                color='red',
                symbol='triangle-down'),
    showlegend=True,
    opacity=0.7,
    name='RSI Sell Signals'
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=rsi_sma_20.index,
    y=rsi_sma_20,
    line=dict(color='orange', width=2),
    opacity=0.7,
    showlegend=True,
    name='SMA 20'
), row=2, col=1)


In [565]:
fig.add_trace(go.Scatter(
    x=btc_stochrsi_values['d'].index,
    y=btc_stochrsi_values['d'],
    line=dict(color='lightcoral', width=2),
    showlegend=True,
    name='StochRSI d'
), row=3, col=1)

fig.add_trace(go.Scatter(
    x=btc_rsi_values.index,
    y=btc_rsi_values,
    line=dict(color='lightblue', width=2),
    showlegend=True,
    name='RSI'
), row=3, col=1)

fig.add_trace(go.Scatter(
    x=btc_stochrsi_values['d'][btc_stochrsi_buy].index,
    y=btc_stochrsi_values['d'][btc_stochrsi_buy],
    mode='markers',
    marker=dict(size=10,
                color='green',
                symbol='triangle-up'),
    showlegend=True,
    opacity=0.7,
    name='StochRSI Buy Signals'
), row=3, col=1)

fig.add_trace(go.Scatter(
    x=btc_stochrsi_values['d'][btc_stochrsi_sell].index,
    y=btc_stochrsi_values['d'][btc_stochrsi_sell],
    mode='markers',
    marker=dict(size=10,
                color='red',
                symbol='triangle-down'),
    showlegend=True,
    opacity=0.7,
    name='StochRSI Sell Signals'
), row=3, col=1)

fig.update_layout()

In [566]:
fig.add_trace(go.Scatter(
    x=btc_bb_values.index,
    y=btc_bb_values,
    line=dict(color='lightblue', width=2),
    showlegend=True,
    name='BB'
), row=4, col=1)

fig.add_trace(go.Scatter(
    x=btc_bb_values[btc_bb_buy].index,
    y=btc_bb_values[btc_bb_buy],
    mode='markers',
    marker=dict(size=10,
                color='green',
                symbol='triangle-up'),
    showlegend=True,
    opacity=0.7,
    name='BB Buy Signals'
), row=4, col=1)

fig.add_trace(go.Scatter(
    x=btc_bb_values[btc_bb_sell].index,
    y=btc_bb_values[btc_bb_sell],
    mode='markers',
    marker=dict(size=10,
                color='red',
                symbol='triangle-down'),
    showlegend=True,
    opacity=0.7,
    name='BB Sell Signals'
), row=4, col=1)

fig.add_trace(go.Scatter(
    x=bb_sma_20.index,
    y=bb_sma_20,
    line=dict(color='orange', width=2),
    opacity=0.7,
    showlegend=True,
    name='BB SMA 20'
), row=4, col=1)

fig.update_layout(xaxis_rangeslider_visible=False)

In [567]:
fig.add_trace(go.Scatter(
    x=btc_macd_values['macd'].index,
    y=btc_macd_values['macd'],
    line=dict(color='lightblue', width=1),
    showlegend=True,
    name='MACD'
), row=5, col=1)

fig.add_trace(go.Scatter(
    x=btc_macd_values['signal'].index,
    y=btc_macd_values['signal'],
    line=dict(color='red', width=1),
    opacity=0.7,
    showlegend=True,
    name='MACD Signal'
), row=5, col=1)

fig.add_trace(go.Bar(
    x=btc_macd_values['histogram'].index,
    y=btc_macd_values['histogram'],
    marker_color=btc_macd_values['histogram'],
    showlegend=True,
    name='MACD Histogram'
), row=5, col=1)

fig.add_trace(go.Scatter(
    x=btc_macd_values['macd'][btc_macd_buy].index,
    y=btc_macd_values['macd'][btc_macd_buy],
    mode='markers',
    marker=dict(size=12,
                color='green',
                symbol='triangle-up'),
    showlegend=True,
    opacity=0.7,
    name='MACD Buy Signals'
), row=5, col=1)

fig.add_trace(go.Scatter(
    x=btc_macd_values['macd'][btc_macd_sell].index,
    y=btc_macd_values['macd'][btc_macd_sell],
    mode='markers',
    marker=dict(size=12,
                color='red',
                symbol='triangle-down'),
    showlegend=True,
    opacity=0.7,
    name='MACD Sell Signals'
), row=5, col=1)

fig.update_layout(xaxis_rangeslider_visible=False)

In [568]:

fig.add_trace(go.Scatter(
    x=btc_sharpe_values.index,
    y=btc_sharpe_values,
    line=dict(color='lightblue', width=1),
    showlegend=True,
    name='Sharpe'
), row=6, col=1)

fig.add_trace(go.Scatter(
    x=btc_sharpe_values[sharpe_buy].index,
    y=btc_sharpe_values[sharpe_buy],
    mode='markers',
    marker=dict(size=btc_sharpe_values[sharpe_buy].abs()*5,
                color='green',
                symbol='circle'),

    showlegend=True,
    opacity=0.5,
    name='Sharpe Buy Signals'
), row=6, col=1)

fig.add_trace(go.Scatter(
    x=btc_sharpe_values[sharpe_sell].index,
    y=btc_sharpe_values[sharpe_sell],
    mode='markers',
    marker=dict(size=btc_sharpe_values[sharpe_sell].abs()*3,
                color='red',
                symbol='circle'),

    showlegend=True,
    opacity=0.7,
    name='Sharpe Sell Signals'
), row=6, col=1)

fig.add_trace(go.Scatter(
    x=btc_sharpe_sma_20.index,
    y=btc_sharpe_sma_20,
    line=dict(color='purple', width=2),
    showlegend=True,
    opacity=0.7,
    name='Sharpe SMA 20'
), row=6, col=1)

fig.add_trace(go.Bar(
    x=btc_sortino_values.index,
    y=btc_sortino_values,
    marker_color=btc_sortino_values,
    showlegend=True,
    opacity=0.7,
    name='Sortino'
), row=6, col=1)

fig.add_trace(go.Scatter(
    x=btc_sortino_values[sortino_buy].index,
    y=btc_sortino_values[sortino_buy],
    mode='markers',
    marker=dict(size=btc_sortino_values[sortino_buy].abs()*5,
                color='green',
                symbol='triangle-up'),

    showlegend=True,
    opacity=0.7,
    name='Sortino Buy Signals'
), row=6, col=1)

fig.add_trace(go.Scatter(
    x=btc_sortino_values[sortino_sell].index,
    y=btc_sortino_values[sortino_sell],
    mode='markers',
    marker=dict(size=btc_sortino_values[sortino_sell].abs()*3,
                color='red',
                symbol='triangle-down'),
    showlegend=True,
    opacity=0.7,
    name='Sortino Sell Signals'
), row=6, col=1)

fig.add_trace(go.Scatter(
    x=btc_sortino_sma_14.index,
    y=btc_sortino_sma_14,
    line=dict(color='#FFBF00', width=2),
    showlegend=True,
    opacity=0.5,
    name='Sortino SMA 14'
), row=6, col=1)

fig.add_trace(go.Scatter(
    x=btc_sortino_sma_7.index,
    y=btc_sortino_sma_7,
    line=dict(color='#FF4500', width=2),
    showlegend=True,
    opacity=0.5,
    name='Sortino SMA 7'
), row=6, col=1)

fig.update_layout(template='plotly_dark')

In [569]:
fig.add_trace(go.Bar(
    x=vix.index,
    y=vix,
    marker_color=vix,
    showlegend=True,
    opacity=0.7,
    name='VIX'
), row=7, col=1)

fig.add_trace(go.Scatter(
    x=vix_sma_9.index,
    y=vix_sma_9,
    marker_color='orange',
    showlegend=True,
    opacity=0.7,
    name='VIX SMA 9'
), row=7, col=1)

fig.add_trace(go.Scatter(
    x=vix_sma_20.index,
    y=vix_sma_20,
    marker_color='lightgreen',
    showlegend=True,
    name='VIX SMA 50'
), row=7, col=1)

In [570]:

class Stock:

    def __init__(self, symbol: str, date_init: str, date_end: str = (datetime.now() + timedelta(days=7)).strftime('%Y-%m-%d'), interval: str = "1d", thresholds=None):
        self.symbol = symbol
        self.date_init = date_init
        self.date_end = date_end or datetime.now() + timedelta(days=7).strftime('%Y-%m-%d')
        self.interval = interval
        self._df: Optional[pd.DataFrame] = None
        self.thresholds = thresholds or {
            "BTC-USD": {
                "sharpe_top": 2.05,
                "sortino_top": 5.65,
                "sharpe_bottom": -1.1,
                "sortino_bottom": -1.7,
            },
            "SPY": {
                "sharpe_top": 2.2,
                "sortino_top": 5,
                "sharpe_bottom": -0.5,
                "sortino_bottom": -1.1,
            }
        }
    @property
    def df(self) -> pd.DataFrame:
        if self._df is None:  # Só baixa se não tiver dados
            self._df = self.create_dataframe()
        return self._df

    def __str__(self):
        return f"Stock(symbol='{self.symbol}', date_init='{self.date_init}', date_end='{self.date_end}', interval='{self.interval}')"

    def create_dataframe(self):
        return yf.download(self.symbol, start=self.date_init, end=self.date_end, interval=self.interval).dropna().copy()

    def create_dataframe_close(self):
        close = self.df["Close"]
        if isinstance(close, pd.DataFrame):
            if self.symbol in close.columns:
                close = close[self.symbol]
            else:
                close = close.iloc[:, 0]
        return close

    def create_dataframe_low(self):
        low = self.df["Low"]
        if isinstance(low, pd.DataFrame):
            if self.symbol in low.columns:
                low = low[self.symbol]
            else:
                low = low.iloc[:, 0]
        return low

    def create_dataframe_high(self):
        high = self.df["High"]
        if isinstance(high, pd.DataFrame):
            if self.symbol in high.columns:
                high = high[self.symbol]
            else:
                high = high.iloc[:, 0]
        return high

    def normalize(self):
        close = self.create_dataframe_close()
        return close / close.iloc[0]

    def log_return(self):
        close = self.create_dataframe_close()
        return np.log(close / close.shift(1)).dropna()

    def return_investment(self):
        acumulated_returns = self.create_dataframe_close().pct_change().dropna()
        return (1 + acumulated_returns).cumprod()

    def sharpe_ratio(self):
        rf = 0.01
        if self.interval == "1mo":
            rolling_mean = self.log_return().rolling(window=6).mean() * 12
            rolling_std = self.log_return().rolling(window=6).std() * np.sqrt(12)
            sharpe_ratio = (rolling_mean - rf) / rolling_std
            return sharpe_ratio.reindex(self.df.index)
        elif self.interval == "1wk":
            rolling_mean = self.log_return().rolling(window=60).mean() * 52
            rolling_std = self.log_return().rolling(window=60).std() * np.sqrt(52)
            sharpe_ratio = (rolling_mean - rf) / rolling_std
            return sharpe_ratio.reindex(self.df.index)

        elif self.interval == "1d":
            rolling_mean = self.log_return().rolling(window=252).mean() * 252
            rolling_std = self.log_return().rolling(window=252).std() * np.sqrt(252)
            sharpe_ratio = (rolling_mean - rf) / rolling_std
            return sharpe_ratio.reindex(self.df.index)
        else:
            return 0

    def sortino_ratio(self):
        returns = self.create_dataframe_close().pct_change().dropna()
        rf = 0.01
        if self.interval == "1mo":
            mar = rf / 12
            excess = returns - mar
            downside = np.minimum(0, excess)
            downside_deviation = downside.rolling(window=14).std() * np.sqrt(12)
            rolling_mean = excess.rolling(window=14).mean() * 12
            sortino_ratio = rolling_mean / downside_deviation
            sortino_ratio = sortino_ratio.replace([np.inf, -np.inf], np.nan)
            return sortino_ratio.reindex(self.df.index)
        elif self.interval == "1wk":
            mar = rf / 52
            excess = returns - mar
            downside = np.minimum(0, excess)
            downside_deviation = downside.rolling(window=60).std() * np.sqrt(52)
            rolling_mean = excess.rolling(window=60).mean() * 52
            sortino_ratio = rolling_mean / downside_deviation
            sortino_ratio = sortino_ratio.replace([np.inf, -np.inf], np.nan)
            return sortino_ratio.reindex(self.df.index)
        if self.interval == "1d":
            mar = rf / 252
            excess = returns - mar
            downside = np.minimum(0, excess)
            downside_deviation = downside.rolling(window=252).std() * np.sqrt(252)
            rolling_mean = excess.rolling(window=252).mean() * 252
            sortino_ratio = rolling_mean / downside_deviation
            sortino_ratio = sortino_ratio.replace([np.inf, -np.inf], np.nan)
            return sortino_ratio.reindex(self.df.index)
        else:
            return 0

    def compute_masks(self):
        shp = self.sharpe_ratio()
        srt = self.sortino_ratio()

        th = self.thresholds[self.symbol]

        masks = {
            "sortino_top": srt > th["sortino_top"],
            "sharpe_top": shp > th["sharpe_top"],
            "sharpe_sortino_top": (srt >= th["sortino_top"]) & (shp > th["sharpe_top"]),

            "sortino_reversion_down": (shp < 1.2) & (srt >= th["sortino_top"]),

            "sharpe_bottom": shp < th["sharpe_bottom"],
            "sortino_bottom": srt < th["sortino_bottom"],
            "sharpe_sortino_bottom": (srt < th["sortino_bottom"]) & (shp < th["sharpe_bottom"]),

            "sortino_reversion_up": (shp > 0) & (srt <= th["sortino_bottom"]),
        }
        return masks

    def ema(self, period: int):
        close = self.create_dataframe_close()
        return close.ewm(span=period, adjust=False).mean()

    def macd(self, fast=12, slow=26, signal=9):
        macd_line = self.ema(fast) - self.ema(slow)
        signal_line = macd_line.ewm(span=signal, adjust=False).mean()
        histogram = macd_line - signal_line
        return macd_line, signal_line, histogram

    def bb_percent(self, window=20, std=2):
        close = self.create_dataframe_close()
        midle = close.rolling(window).mean()
        upper = midle + (close.rolling(window).std() * std)
        lower = midle - (close.rolling(window).std() * std)
        return (close - lower) / (upper - lower)

    def rsi(self, period: int = 14):
        close = self.create_dataframe_close()
        delta = close.diff()
        gain = (delta.where(delta > 0, 0)).rolling(period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(period).mean()
        rs = gain / loss
        rsi = 100 - (100 / (1 + rs))
        return rsi


    def stoch_rsi(self, rsi_period: int = 14, stoch_period: int = 14, smooth_k: int = 3, smooth_d: int = 3):
        rsi = self.rsi(rsi_period)

        rsi_low = rsi.rolling(stoch_period).min()
        rsi_high = rsi.rolling(stoch_period).max()

        denom = (rsi_high - rsi_low).replace(0, np.nan)
        k_raw = (rsi - rsi_low) / denom * 100

        k = k_raw.rolling(window=smooth_k).mean() if smooth_k and smooth_k > 1 else k_raw
        d = k.rolling(window=smooth_d).mean() if smooth_d and smooth_d > 1 else k

        return k.dropna(), d.dropna()

@dataclass
class Signal(ABC):

    @abstractmethod
    def get_sinal_top(self) -> bool:
        pass

    @abstractmethod
    def get_signal_bottom(self) -> bool:
        pass


In [571]:
# Exemplo de uso
btc_w = Stock("BTC-USD", "2018-01-01", interval="1wk")
vix_w = Stock("^VIX", "2018-01-01", interval="1wk")
spy_w = Stock("SPY", "2018-01-01", interval="1wk")
eth_w = Stock("ETH-USD", "2018-01-01", interval="1wk")

In [572]:
eth_w.thresholds['ETH-USD'] = {'sharpe_top': 1.7, 'sortino_top': 5.65, 'sharpe_bottom': -1.1, 'sortino_bottom': -1.7}
vix_w.thresholds['^VIX'] = {'sharpe_top': 0.5, 'sortino_top': 2, 'sharpe_bottom': -0.5, 'sortino_bottom': -1}

In [573]:
btc_w.df

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,BTC-USD,BTC-USD,BTC-USD,BTC-USD,BTC-USD
Date,,,,,
2018-01-01,16477.599609,17712.400391,13154.700195,14112.200195,123814400000
2018-01-08,13772.000000,16537.900391,13105.900391,16476.199219,106022199296
2018-01-15,11600.099609,14445.500000,9402.290039,13767.299805,97932879872
2018-01-22,11786.299805,12040.299805,10129.700195,11633.099609,64691999232
2018-01-29,8277.009766,11875.599609,7796.490234,11755.500000,60810019840
...,...,...,...,...,...
2026-03-09,72789.914062,73927.328125,65858.007812,65969.585938,301054621121
2026-03-16,67845.210938,75988.398438,67372.875000,72798.171875,285450089323


In [574]:
btc_mask = btc_w.compute_masks()
spy_mask = spy_w.compute_masks()
eth_mask = eth_w.compute_masks()
vix_mask = vix_w.compute_masks()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [575]:
fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Scatter(
    x=btc_w.ema(20).index,
    y=btc_w.ema(20)
))
fig.add_trace(go.Scatter(
    x=btc_w.ema(50).index,
    y=btc_w.ema(50)
))
fig.add_trace(go.Scatter(
    x=btc_w.ema(100).index,
    y=btc_w.ema(100)
))
fig.add_trace(go.Scatter(
    x=btc_w.ema(200).index,
    y=btc_w.ema(200)
))
f()

NameError: name 'f' is not defined

In [ ]:
sharpe_top_dates_spy = spy_w.sharpe_ratio().index[spy_mask["sharpe_top"]]
sharpe_top_values_spy = spy_w.sharpe_ratio()[spy_mask["sharpe_top"]]

fig_spy_ss = make_subplots(rows=2, cols=1)

fig_spy_ss.add_trace(go.Scatter(
    x=spy_w.sharpe_ratio().index,
    y=spy_w.sharpe_ratio(),
    marker_color='white')
    , row=1, col=1)
fig_spy_ss.add_trace(go.Bar(
    x=spy_w.sortino_ratio().index,
    y=spy_w.sortino_ratio(),
    marker_color=spy_w.sortino_ratio())
    , row=1, col=1)
fig_spy_ss.add_trace(go.Scatter(
    x=sharpe_top_dates_spy,
    y=sharpe_top_values_spy,
    mode='markers',
    marker=dict(size=sharpe_top_values_spy*5, color='red', symbol='diamond'),
), row=1, col=1)

fig_spy_ss.update_layout(title="Sharpe and Sortino Ratios", xaxis_title="Date", yaxis_title="Ratio", template="plotly_dark")
fig_spy_ss

In [ ]:
sharpe_top_dates_btc = btc_w.sharpe_ratio().index[btc_mask["sharpe_top"]]
sharpe_top_values_btc = btc_w.sharpe_ratio()[btc_mask["sharpe_top"]]

fig_btc_ss = make_subplots(rows=1, cols=1)

fig_btc_ss.add_trace(go.Scatter(
    x=btc_w.sharpe_ratio().index,
    y=btc_w.sharpe_ratio(),
    marker_color='white')
    , row=1, col=1)
fig_btc_ss.add_trace(go.Bar(
    x=btc_w.sortino_ratio().index,
    y=btc_w.sortino_ratio(),
    marker_color=btc_w.sortino_ratio())
    , row=1, col=1)
fig_btc_ss.add_trace(go.Scatter(
    x=sharpe_top_dates_btc,
    y=sharpe_top_values_btc,
    mode='markers',
    marker=dict(size=sharpe_top_values_btc*5, color='red', symbol='diamond'),
), row=1, col=1)

fig_btc_ss.update_layout(title="Sharpe and Sortino Ratios", xaxis_title="Date", yaxis_title="Ratio", template="plotly_dark")
fig_btc_ss.show()

In [ ]:
sharpe_top_dates_vix = vix_w.sharpe_ratio().index[vix_mask["sharpe_top"]]
sharpe_top_values_vix = vix_w.sharpe_ratio()[vix_mask["sharpe_top"]]

fig_vix_ss = make_subplots(rows=2, cols=1)

fig_vix_ss.add_trace(go.Scatter(
    x=vix_w.sharpe_ratio().index,
    y=vix_w.sharpe_ratio(),
    marker_color='white')
    , row=1, col=1)
fig_vix_ss.add_trace(go.Bar(
    x=vix_w.sortino_ratio().index,
    y=vix_w.sortino_ratio(),
    marker_color=vix_w.sortino_ratio())
    , row=1, col=1)
fig_vix_ss.add_trace(go.Scatter(
    x=sharpe_top_dates_vix,
    y=sharpe_top_values_vix,
    mode='markers',
    marker=dict(size=sharpe_top_values_vix*20, color='red', symbol='diamond'),
), row=1, col=1)

fig_vix_ss.update_layout(title="Sharpe and Sortino Ratios", xaxis_title="Date", yaxis_title="Ratio", template="plotly_dark")
fig_vix_ss

In [ ]:
sharpe_top_dates_eth = eth_w.sharpe_ratio().index[eth_mask["sharpe_top"]]
sharpe_top_values_eth = eth_w.sharpe_ratio()[eth_mask["sharpe_top"]]

fig_eth_ss = make_subplots(rows=2, cols=1)

fig_eth_ss.add_trace(go.Scatter(
    x=eth_w.sharpe_ratio().index,
    y=eth_w.sharpe_ratio(),
    marker_color='white')
    , row=1, col=1)
fig_eth_ss.add_trace(go.Bar(
    x=eth_w.sortino_ratio().index,
    y=eth_w.sortino_ratio(),
    marker_color=eth_w.sortino_ratio())
    , row=1, col=1)
fig_eth_ss.add_trace(go.Scatter(
    x=sharpe_top_dates_eth,
    y=sharpe_top_values_eth,
    mode='markers',
    marker=dict(size=sharpe_top_values_eth*4, color='red', symbol='diamond'),
), row=1, col=1)

fig_eth_ss.update_layout(title="Sharpe and Sortino Ratios", xaxis_title="Date", yaxis_title="Ratio", template="plotly_dark")
fig_eth_ss

In [ ]:
macd_line, signal_line, histogram = btc_w.macd()

fig3 = make_subplots(rows=1, cols=1)

fig3.add_trace(go.Bar(
    x=histogram.index,
    y=histogram,
    name='Histogram',
    marker_color=histogram
))

fig3.add_trace(go.Scatter(
    x=macd_line.index,
    y=macd_line,
    name='MACD Line',
    line=dict(color='#2962FF')
))

fig3.add_trace(go.Scatter(
    x=signal_line.index,
    y=signal_line,
    name='Signal Line',
    line=dict(color='#FF6D00')
))

fig3.update_layout(title='MACD - BTC', xaxis_title='Date', yaxis_title='Value', template=('plotly_dark'))
fig3.show()

In [ ]:
fig4 = make_subplots(rows=1, cols=1)

fig4.add_trace(go.Scatter(
    x=btc_w.bb_percent().index,
    y=btc_w.bb_percent()
))
fig4.show()

In [ ]:
stoch_k, stoch_d = btc_w.stoch_rsi()
rsi = btc_w.rsi()

fig5 = make_subplots(rows=1, cols=1)

fig5.add_trace(go.Scatter(
    x=stoch_d.index,
    y=stoch_d))

fig5.add_trace(go.Scatter(
    x=rsi.index,
    y=rsi))

fig5.show()

RSI + STOCHASTICO OS DOIS JUNTOS NAS EXTREMIDADES É INVERSAO DE TENDENCIA